# 03_条件路由模式

## 条件路由：让图自己决定下一步

条件边、 Command 路由、多种路由模式 · 第三课 · 约 40 分钟

### 目录

-   回顾：简单边
-   一、从一条直线到十字路口
-   二、方式一：add\_conditional\_edges
-   三、方式二：Command（推荐）
-   四、两种方式对比
-   五、经典路由模式
-   六、动手跑一下
-   七、常见坑
-   八、总结

### 回顾：简单边

前两课我们只用了一种边——**简单边**（add\_edge），它的特点：

-   「做完 A → 下一步一定是 B」
-   固定路线，没有选择

**简单边 = 确定性的下一步。**但如果下一步取决于当前状态呢？比如“是否需要主管审批”的判断结果？

这就是**条件边**出场的时候了。

### 一、从一条直线到十字路口

想象一个**审批机器人**：

- 员工提交请假申请
- 系统/模型判断：这单是否满足“自动通过”规则？
- 如果能自动通过 → 走自动审批节点
- 否则 → 转主管审批节点

```text
flowchart LR
    classDef startEnd fill:#f59e0b,stroke:#d97706,color:#fff
    classDef process fill:#93c5fd,stroke:#60a5fa,color:#fff
    classDef auto fill:#86efac,stroke:#22c55e,color:#166534
    classDef manager fill:#fca5a5,stroke:#ef4444,color:#991b1b
    START([START]) --> 分类(分类<br>规则/LLM 判断)
    分类 --> 自动通过(auto_approve)
    自动通过 --> END([END])
    分类 --> 主管审批(manager_review)
    主管审批 --> END
    class START,END startEnd
    class 分类 process
    class 自动通过 auto
    class 主管审批 manager
```

**关键在于：**「分类节点」之后没有固定的下一步——它要根据判断结果来**动态选择**。

在 LangGraph 中，有两种方式实现这种「动态选择」：

-   add\_conditional\_edges 经典方式——路由函数与节点分离
-   Command 推荐方式——路由和状态更新在一起

### 二、方式一：add\_conditional\_edges

这是最「原始」的方式。核心思路：

1.  节点只负责处理数据和返回 State 更新
2.  路由函数单独定义，读取 State，返回下一节点名称
3.  用 add\_conditional\_edges 把路由函数绑定到节点

**代码结构**



In [ ]:
from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    request: str
    intent: str
    result: str


# 1. 节点：只负责产出「增量更新」

def classify(state: State):
    req = state["request"]

    # 用规则/LLM 判断是否能自动通过（这里用简单规则模拟）
    if "1天" in req or "1 天" in req or "半天" in req:
        return {"intent": "auto"}
    return {"intent": "manager"}


def auto_approve(state: State):
    return {"result": "自动通过：已进入归档流程"}


def manager_review(state: State):
    return {"result": "转主管审批：等待人工确认"}


# 2. 路由函数：单独定义，只读 State，返回下一节点名称

def router(state: State) -> Literal["auto_approve", "manager_review", "__end__"]:
    if state["intent"] == "auto":
        return "auto_approve"
    if state["intent"] == "manager":
        return "manager_review"
    return "__end__"


# 3. 构建图
builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("auto_approve", auto_approve)
builder.add_node("manager_review", manager_review)

builder.add_edge(START, "classify")
builder.add_edge("auto_approve", END)
builder.add_edge("manager_review", END)

# 4. 关键：用 add_conditional_edges 绑定路由
builder.add_conditional_edges(
    "classify",  # 从哪个节点出发
    router,       # 路由函数——决定下一步去哪
)


> 💡 **Note**  
> 返回值和节点名对应关系：
> 
> 路由函数返回的字符串就是目标节点的名称。
> 
> "**end**"
> 
> 是特殊值，表示直接结束。

**path\_map 参数**

如果你不想让路由函数的返回值直接等于节点名（比如返回 True/False），可以用 `path_map` 做映射：



In [2]:
builder.add_conditional_edges(
    "classify",
    lambda state: state["intent"] == "question",  # 返回 True/False
    {True: "answer", False: "transfer"}       # 映射表
)


> **❓ 检查理解 ①**  
> 一个节点上可以同时有 `add_edge` 和 `add_conditional_edges` 吗？
> 
> -   A. 可以。一个节点可以既有简单边又有条件边
> -   B. 不行。一个节点只能有一个「出口」
> -   C. 可以，但简单边会被条件边覆盖

> **✅ 答案：A**
> 
> 可以共存。但要注意：共存时两条边都会触发，简单边目标和路由函数返回的目标会同时执行（并行），并不是谁覆盖谁。所以实际开发中一个节点的出口通常只选一种方式。

> **高级用法：一个节点同时使用 add\_edge + add\_conditional\_edges**  
> 在同一个节点上同时添加普通边和条件边是允许的：
> 
> ```python
> builder.add_conditional_edges(
>     "llm",
>     should_continue,
>     {"tool": "tool", "end": END}
> )
> builder.add_edge("llm", "audit")   # 同时存在
> ```
> 
> 此时 `llm` 节点执行完毕后，**两条路径都会被触发**，`tool` 和 `audit` 节点会在下一个超步中**并行执行**。
> 
> **适用场景**：
> 
> -   所有请求都需要记录审计日志，同时还要根据条件进行业务分流
> -   节点执行后既要触发主逻辑，又要无条件触发一个辅助任务
> 
> **注意事项**：
> 
> -   这种配置会让图的拓扑结构变复杂，容易出现意外的并发更新。
> -   必须确保 State 的 reducer（如 operator.add）能够正确处理多个节点并行更新时产生的并发写入。
> -   推荐做法：尽量让一个节点的出口只使用一种方式（优先使用 Command，或只用条件边），避免混用。

### 三、方式二：Command（推荐）

LangGraph 提供了更现代的方式——**Command**。核心思路：**路由决策写在节点函数内部**。

> **类比：工位增加「分流功能」**  
> 之前：工位只管干活，由传送带末端的分流器来决定下一站。
> 
> Command：工位自己多了一个按钮——干完活后，自己按一下，选择把产品送到哪里。
> 
> 每个节点函数不仅可以返回 State 更新，还能同时告诉框架「下一步去哪」。

**代码结构**



In [3]:
from typing import Literal

from langgraph.types import Command


# 节点同时返回「更新」和「下一步」

def classify_node(state: State) -> Command[Literal["auto_approve", "manager_review"]]:
    # 通常这里会调用规则/LLM（这里沿用上一个例子的规则）
    req = state["request"]
    intent = "auto" if ("1天" in req or "1 天" in req or "半天" in req) else "manager"

    return Command(
        update={"intent": intent},
        goto=["auto_approve"] if intent == "auto" else ["manager_review"],
    )


不需要 `add_conditional_edges`，直接 `add_edge` 就行：



In [4]:
from langgraph.graph import StateGraph, START, END

builder_cmd = StateGraph(State)

builder_cmd.add_node("classify", classify_node)
builder_cmd.add_node("auto_approve", auto_approve)
builder_cmd.add_node("manager_review", manager_review)

builder_cmd.add_edge(START, "classify")
# 不需要 add_conditional_edges：Command 自己通过 goto 完成路由
builder_cmd.add_edge("auto_approve", END)
builder_cmd.add_edge("manager_review", END)

graph_cmd = builder_cmd.compile()


**Command 的完整用法**



In [5]:
from langgraph.types import Command

# Command 的两个关键参数：update（状态增量）与 goto（下一跳节点）
example = Command(
    update={"intent": "auto"},
    goto=["auto_approve"],
)

example

Command(update={'intent': 'auto'}, goto=['auto_approve'])


`goto` 可以是一个节点名，也可以是节点列表（并行执行）。

> **❓ 检查理解 ②**  
> Command 方式和 add\_conditional\_edges 方式比，最大的区别是什么？
> 
> -   A. Command 性能更好
> -   B. Command 把路由逻辑和节点逻辑放在一起，代码更集中
> -   C. Command 只能走一个节点，add\_conditional\_edges 可以走多个

> **✅ 答案：B**
> 
> Command 最核心的优势就是路由和逻辑集中在一个函数里。路由逻辑通常依赖节点的计算结果（比如 LLM 分类结果），放在一起更自然。

### 四、两种方式对比

以后**优先用 Command**——它更现代，更直观。但阅读别人的代码时，两种都要看得懂。

| 维度  | add\\_conditional\\_edges | Command |
| --- | --- | --- |
| 路由逻辑位置 | 单独的函数 | 在节点函数内部 |
| 代码组织 | 分离（路由与逻辑解耦） | 集中（一个函数搞定） |
| 适用场景 | 路由逻辑简单，只读 State | 路由依赖节点内部计算（如 LLM 结果） |
| 同时更新 State + 路由 | 需要两步（节点返回 + 路由函数） | 一次 Command 搞定 |
| 并行路由 | 支持（返回多个节点名） | 支持（goto 传列表） |
| 可读性 | 结构清晰，但分散 | 更直观，一个节点一个函数 |

**什么时候用哪个？**

**❌ add\_conditional\_edges 路由逻辑简单**



In [6]:
def router(state):
    return "b" if state["x"] > 0 else "c"

builder.add_conditional_edges(
    "a", router
)


**✅ Command 路由依赖节点内部计算结果**



In [7]:
def node_a(state):
    result = llm.invoke(...)    # 内部计算
    return Command(
        update={"analysis": result},
        goto=["b" if result.is_ok else "c"]
    )


### 五、经典路由模式

条件路由不止一种用法。以下是最常见的几种模式，都是实际项目里反复出现的套路。

**模式 1：分类路由（Category Router）**

最基础的模式：LLM 分类 → 根据分类走不同路径。



In [8]:
def classify_intent(state):
    # 调用 LLM 对输入做分类（示意）
    prompt = f"把以下申请分类：\n{state['query']}\n类别：请假/加班/出差/其他"
    category = llm.invoke(prompt).content.strip()

    # 根据分类路由到不同的处理节点
    route_map = {
        "请假": "leave_handler",
        "加班": "overtime_handler",
        "出差": "travel_handler",
    }
    next_node = route_map.get(category, "general_handler")

    return Command(update={"category": category}, goto=[next_node])


**模式 2：质量门（Quality Gate）**

先做一遍，检查结果质量，不合格就重做或走备选。



In [9]:
def validation_gate(state):
    # 检查上一个节点的输出质量
    draft = state["draft"]
    is_valid = len(draft) > 10 and "错误" not in draft

    if is_valid:
        return Command(update={"validated": True}, goto=["send"])
    else:
        # 质量不合格，重新生成
        return Command(update={"validated": False}, goto=["regenerate"])


```text
flowchart LR
    classDef startEnd fill:#f59e0b,stroke:#d97706,color:#fff
    classDef process fill:#93c5fd,stroke:#60a5fa,color:#fff
    classDef regenerate fill:#fbbf24,stroke:#d97706,color:#92400e
    classDef send fill:#86efac,stroke:#22c55e,color:#166534
    START([START]) --> 生成草稿(生成草稿<br>LLM)
    生成草稿 --> 重新生成(重新生成)
    重新生成 --> 生成草稿
    生成草稿 --> 发送(发送)
    发送 --> END([END])
    class START,END startEnd
    class 生成草稿 process
    class 重新生成 regenerate
    class 发送 send
```

**模式 3：退避/兜底（Fallback Router）**

主路径失败时走备选路径。适合调用外部 API 的场景。



In [ ]:
def try_primary(state):
    try:
        result = call_expensive_llm(state["query"])
        return Command(
            update={"result": result},
            goto=["format_output"]     # 成功 → 正常路径
        )
    except Exception:
        return Command(
            update={"result": ""},
            goto=["fallback_handler"]   # 失败 → 兜底路径
        )


**模式 4：自循环（Self-loop）**

同一个节点反复执行，直到满足条件。比如：反复追问用户，直到收集到完整信息。



In [11]:
def is_info_complete(state):
    # 示例：真实项目里会检查 state 是否已收集到必要字段
    return bool(state.get("employee_id"))


def collect_info(state):
    if is_info_complete(state):
        return Command(update={}, goto=["处理"])

    question = "请补充您的工号（employee_id）用于继续审批："
    return Command(
        update={"last_question": question},
        goto=["信息收集"],  # 回到自己
    )


# 构建时：信息收集节点可以连到自身
builder.add_node("信息收集", collect_info)
builder.add_edge(START, "信息收集")


> **❓ 检查理解 ③**  
> 以下哪种情况**最适合**用 Fallback Router（兜底模式）？
> 
> -   A. 根据用户输入走 A 或 B
> -   B. 调用一个不稳定 API，失败时走备选方案
> -   C. 并行执行两个搜索节点，然后合并结果

> **✅ 答案：B**
> 
> Fallback Router 就是为「主路径可能失败，需要备选」的场景设计的。

### 六、动手跑一下

**Demo 1：分类路由（Command 方式）**

一个完整可运行的审批分类器。保存为 `demo_router.py`，在 conda 环境里运行 `python demo_router.py`：



In [12]:
from typing import TypedDict, Literal

from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


# ---- State ----
class State(TypedDict):
    request: str
    intent: str
    result: str


# ---- 节点 1：分类（用 Command 路由）----
def classify(state: State) -> Command[Literal["auto", "manager", "other"]]:
    print(f"[分类] 收到: {state['request']}")
    req = state["request"]

    if "请假" in req and ("半天" in req or "1天" in req or "1 天" in req):
        intent, next_node = "auto", "auto"
    elif "请假" in req:
        intent, next_node = "manager", "manager"
    else:
        intent, next_node = "other", "other"

    return Command(update={"intent": intent}, goto=[next_node])


# ---- 节点 2/3/4：处理节点 ----
def auto(state: State):
    print("[自动] 命中自动审批规则")
    return {"result": "已自动通过（待归档）"}


def manager(state: State):
    print("[主管] 转主管审批")
    return {"result": "已提交主管审批（等待确认）"}


def other(state: State):
    print("[其他] 转人工处理")
    return {"result": "已转人工处理，请补充说明。"}


# ---- 构建图 ----
builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("auto", auto)
builder.add_node("manager", manager)
builder.add_node("other", other)

builder.add_edge(START, "classify")
builder.add_edge("auto", END)
builder.add_edge("manager", END)
builder.add_edge("other", END)

graph = builder.compile()


# ---- 测试 ----
test_cases = [
    "我想请假半天，原因：感冒",
    "我想请假 3 天，原因：回老家",
    "你们公司几点上班？",
]

for req in test_cases:
    result = graph.invoke({"request": req, "intent": "", "result": ""})
    print(f"  → [{result['intent']}] {result['result']}\n")

[分类] 收到: 我想请假半天，原因：感冒
[自动] 命中自动审批规则
  → [auto] 已自动通过（待归档）

[分类] 收到: 我想请假 3 天，原因：回老家
[主管] 转主管审批
  → [manager] 已提交主管审批（等待确认）

[分类] 收到: 你们公司几点上班？
[其他] 转人工处理
  → [other] 已转人工处理，请补充说明。




运行后你应该看到：

```text
[分类] 收到: 我想请假半天，原因：感冒
[自动] 命中自动审批规则
  → [auto] 已自动通过（待归档）

[分类] 收到: 我想请假 3 天，原因：回老家
[主管] 转主管审批
  → [manager] 已提交主管审批（等待确认）

[分类] 收到: 你们公司几点上班？
[其他] 转人工处理
  → [other] 已转人工处理，请补充说明。
```

**Demo 2：质量门 + 重试循环**

这个例子演示 Self-loop 模式——一个节点自循环直到满足条件。保存为 `demo_quality_gate.py`：



In [13]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

class State(TypedDict):
    input_text: str
    output: str
    attempts: int

# 生成回复，并检查质量。不合格就自己重做
def generate(state: State) -> Command[Literal["generate", "finalize"]]:
    attempts = state["attempts"] + 1
    print(f"第 {attempts} 次尝试生成...")

    # 模拟生成：假装前两次质量不合格
    output = "简短回复"  # 太短了，不合格
    if attempts >= 2:
        output = "这是一条足够长、表述也合格的回复，可以发送给用户。"

    # 质量检查
    is_valid = len(output) > 10 and "错误" not in output
    max_attempts = attempts >= 3

    if is_valid or max_attempts:
        return Command(
            update={"output": output, "attempts": attempts},
            goto=["finalize"]
        )
    else:
        return Command(
            update={"attempts": attempts},
            goto=["generate"]  # 回自己，重试
        )

def finalize(state: State):
    print(f"最终输出 (尝试了 {state['attempts']} 次): {state['output']}")
    return {}

builder = StateGraph(State)
builder.add_node("generate", generate)
builder.add_node("finalize", finalize)
builder.add_edge(START, "generate")
builder.add_edge("finalize", END)

graph = builder.compile()

result = graph.invoke({"input_text": "你好", "output": "", "attempts": 0})

第 1 次尝试生成...
第 2 次尝试生成...
最终输出 (尝试了 2 次): 这是一条足够长、表述也合格的回复，可以发送给用户。



运行后：

```text
第 1 次尝试生成...
第 2 次尝试生成...
最终输出 (尝试了 2 次): 这是一条足够长、表述也合格的回复，可以发送给用户。
```

**关键观察：**`generate` 节点通过 Command 回到自己，循环执行，直到质量达标。

> 💡 **Note**  
> 小心无限循环！
> 
> 自循环一定要有
> 
> 退出条件
> 
> （如上例的
> 
> max\_attempts
> 
> ）。否则可能无限循环下去。

> **一个真实的调试故事（本教程初版的 bug）**  
> 这个 Demo 初版生成的文本是：`"这是一个足够长且没有包含「错误」字样的合格回复。"`——看出问题了吗？
> 
> 这句话**自己就包含「错误」两个字**，于是质量检查 `"错误" not in output` 永远不通过，循环只能靠 `max_attempts` 兜底退出，跑出来是「尝试了 3 次」而不是预期的 2 次。
> 
> \*\*教训：\*\*① 写质量门时，小心检查条件被「描述检查条件的文本」本身触发（自指陷阱）；② 这正是 `max_attempts` 兜底的价值——就算检查逻辑有 bug，循环也不会失控，只是多跑了一次。

**Demo 3：ReAct 路由（工具调用）**

这是 LangGraph 最经典的路由场景——**LLM 决定是否调用工具**。为了让你**不用 API Key 也能跑通**，我们先用一个「假 LLM」模拟决策逻辑；文末再给出真 LLM 的替换写法。

保存为 `demo_react_router.py`，在 conda 环境里运行 `python demo_react_router.py`：



In [14]:
from typing import Literal
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Command
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

# 1. 定义工具
@tool
def get_weather(city: str):
    """查询城市天气"""
    return f"{city} 今天晴天，25°C"

# 2. 「假 LLM」节点 + Command 路由
#    决策逻辑和真 LLM 完全一致：没有工具结果就发起调用，有了就总结回答
def llm_node(state: MessagesState) -> Command[Literal["tools", END]]:
    last = state["messages"][-1]
    if isinstance(last, ToolMessage):
        # 已拿到工具结果 → 生成最终回答，结束
        answer = AIMessage(content=f"查到了：{last.content}")
        return Command(goto=[END], update={"messages": [answer]})
    # 还没调工具 → 发起 tool_calls（真 LLM 是自己决定这一步）
    call = AIMessage(content="", tool_calls=[
        {"name": "get_weather", "args": {"city": "北京"}, "id": "call_1"}
    ])
    return Command(goto=["tools"], update={"messages": [call]})

# 3. 构建图
builder = StateGraph(MessagesState)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode([get_weather]))
builder.add_edge(START, "llm")
builder.add_edge("tools", "llm")  # 工具执行完回到 LLM

graph = builder.compile()

# 4. 运行
result = graph.invoke({"messages": [HumanMessage(content="北京天气怎么样？")]})
print(f"共 {len(result['messages'])} 条消息:")
for m in result["messages"]:
    role = type(m).__name__
    content = m.content or f"[发起工具调用: {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})]"
    print(f"  {role}: {content}")

共 4 条消息:
  HumanMessage: 北京天气怎么样？
  AIMessage: [发起工具调用: get_weather({'city': '北京'})]
  ToolMessage: 北京 今天晴天，25°C
  AIMessage: 查到了：北京 今天晴天，25°C



运行后你应该看到（实测输出）：

```text
共 4 条消息:
  HumanMessage: 北京天气怎么样？
  AIMessage: [发起工具调用: get_weather({'city': '北京'})]
  ToolMessage: 北京 今天晴天，25°C
  AIMessage: 查到了：北京 今天晴天，25°C
```

\*\*关键观察：\*\*消息序列完整呈现了 ReAct 循环——提问 → LLM 发起工具调用（路由到 tools）→ 工具结果 → LLM 总结回答（路由到 END）。`ToolNode` 自动执行了 `tool_calls` 并把结果包装成 `ToolMessage`。

**换成真 LLM**

装好 `langchain-openai` 并配置 `OPENAI_API_KEY` 后，只需把「假 LLM」的决策部分换掉，其余**一行不改**：



In [15]:
import os

if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini")
    llm_with_tools = llm.bind_tools([get_weather])

    def llm_node(state: MessagesState) -> Command[Literal["tools", END]]:
        result = llm_with_tools.invoke(state["messages"])
        # LLM 自己决定是否调用工具
        if result.tool_calls:
            return Command(goto=["tools"], update={"messages": [result]})
        return Command(goto=[END], update={"messages": [result]})
else:
    print("未检测到 OPENAI_API_KEY，跳过真 LLM 示例（上面的假 LLM 版本可直接运行）。")

未检测到 OPENAI_API_KEY，跳过真 LLM 示例（上面的假 LLM 版本可直接运行）。



这就是 ReAct Agent 的简化版：LLM 决定是否调用工具，需要就转 tools，执行完回 LLM，直到 LLM 认为可以直接回答。

**真实运行解析：4 条消息背后发生了什么**

一位读者用真 LLM（并把工具换成了真实数据源）跑出了下面的结果，我们逐条拆解——这是理解 ReAct 路由最好的素材：

```text
共 4 条消息:
  HumanMessage: 北京天气怎么样？
  AIMessage: [发起工具调用: get_weather({'city': '北京'})]
  ToolMessage: 北京 今天晴天，35°C
  AIMessage: 北京今天晴天，气温35°C。天气炎热，请注意防暑降温，
             多喝水，避免长时间在户外活动。
```

实际执行路径：`START → llm(第1次) → tools → llm(第2次) → END`

| 消息  | 谁产生的 | 背后发生了什么 |
| --- | --- | --- |
| ① HumanMessage | 用户  | 图从 START 进入 llm 节点 |
| ② AIMessage（tool\\_calls） | LLM 第 1 次调用 | LLM 发现自己没有实时天气数据，但 bind\\_tools 告诉它有 get\\_weather 可用 → 它自己从自然语言里抽取出参数 {'city': '北京'} 并发起调用。注意这条消息 content 为空，有效载荷在 tool\\_calls 字段里——我们的路由代码 if result.tool\\_calls 检测到它，走 goto=\\["tools"\\] |
| ③ ToolMessage | ToolNode（纯代码） | ToolNode 读取 tool\\_calls，真正执行 Python 函数，把返回值包装成 ToolMessage。这一步没有任何 AI 参与，是确定性执行。然后走普通边 tools → llm 回到 LLM |
| ④ AIMessage（最终回答） | LLM 第 2 次调用 | 这次历史里已有工具结果，LLM 判断信息足够 → 生成回答，响应里不带 tool\\_calls → 路由走 goto=\\[END\\]，循环终止 |

**三个值得注意的观察：**

-   终止条件是「涌现」的，不是硬编码的。假 LLM 版靠 isinstance(last, ToolMessage) 死规则终止；真 LLM 版靠模型自己判断「我可以回答了」（第二次响应不带 tool\_calls）。如果它觉得信息还不够，完全可以再次发起调用——tools → llm 这条回边就是为多轮调用预留的。
-   最终回答是「接地」（grounded）的。ToolMessage 说 35°C，最终回答也说 35°C——LLM 忠实采用了工具结果，而非凭训练数据瞎编；并且它还增值了：根据 35°C 自己推理出「防暑降温、多喝水」。这是假 LLM 的 f"查到了：{...}" 复读机做不到的。
-   ToolMessage 是消息链里「可信」的一环。它的内容就是工具函数的直接返回值，LLM 无法篡改。AI 只能决定「调不调、怎么总结」，不能伪造工具的返回——如果你看到最终回答的数字和 ToolMessage 对不上，那是模型在幻觉，值得警惕。

> **核心洞察：智能在节点里，结构在图里**
> 
> | 环节  | 假 LLM 版 | 真 LLM 版 |
> | --- | --- | --- |
> | 参数抽取 | 硬编码 {"city": "北京"} | 模型从自然语言里自己抽 |
> | 是否调工具 | isinstance 死规则 | 模型自主决策（tool\\_calls 有无） |
> | 最终回答 | 复读「查到了：...」 | 综合工具结果 + 增值建议 |
> | 图结构 / 路由代码 | 完全相同——换掉大脑，骨架一行不用改 |     |

### 七、常见坑

> **坑 1：Command 的类型提示不匹配**  
> Command 的类型参数**必须**与实际 goto 的节点名一致：
> 
> ```python
> # 正确
> def my_node() -> Command[Literal["a", "b"]]:
>     return Command(goto=["a"])
> 
> # 错误：类型提示写了 "a"|"b"，但实际 goto "c"
> ```
> 
> 类型提示的作用：让 Python 的 Type Checker 发现你在跳转一个不存在的节点。

> **坑 2：自循环忘记退出条件**  
> 节点自己连自己时，一定要有退出条件：
> 
> 🟢 正确：`if 完成: goto("finalize") else: goto("self")`
> 
> 🔴 错误：没有退出条件 → 无限循环 → 程序卡死

> **坑 3：add\_conditional\_edges 和 Command 混用**  
> Command 方式不需要 `add_conditional_edges`。如果对同一个节点两种都用了，两边的目标**都会触发**（并行执行），容易引发意外的并行和 State 写冲突（第二课的 `InvalidUpdateError`）。建议**一个节点的出口只用一种方式，一个图里风格统一**。

### 八、总结

| 概念  | 一句话 |
| --- | --- |
| 条件边 | 让图在运行时根据 State 动态选择下一步 |
| add\\_conditional\\_edges | 经典方式：路由函数和节点分离（适合简单路由） |
| Command | 推荐方式：路由逻辑写在节点内部（适合路由依赖节点计算结果） |
| 分类路由 | LLM 分类后走不同路径 |
| 质量门 | 检查结果质量，不合格就重做 |
| Fallback | 主路径失败时走备选 |
| Self-loop | 自循环直到满足条件（必须有退出条件！） |

> **一句话总结**  
> **简单边 = 指令**（做完 A 必须去 B）
> 
> **条件边 = 决策**（做完 A 后，看一眼结果，再决定去哪）
> 
> **Command = 让节点自己决定**（节点函数内同时返回「更新了什么」和「下一步去哪」）

> **下一课预告**  
> 图会自己选路了，但每次 invoke 还是「失忆」的——跑完就忘。第四课我们学**持久化记忆**：Checkpointer 让图自动存档，thread\_id 区分会话，还能做“时间旅行”回到任意历史状态。它也是后面 Human-in-the-Loop 的地基。

📖 参考：

-   [LangGraph 官方文档 - 条件边](https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges)
-   [Command 文档](https://docs.langchain.com/oss/python/langgraph/graph-api#command)